In [10]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

e:\Codes\german_tutor\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hridi\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [9]:
text = "Ich wohne in der Nähe des Waldes."

tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", ids)
print("Number of token IDs:", len(ids))

inputs = tokenizer(text, return_tensors="pt")
print("Input:", inputs)
print(inputs['input_ids'].shape)

Tokens: ['Ich', 'Ġw', 'oh', 'ne', 'Ġin', 'Ġder', 'ĠNÃ¤he', 'Ġdes', 'ĠWald', 'es', '.']
Number of tokens: 11
Token IDs: [40369, 289, 2267, 811, 304, 2694, 137900, 939, 50568, 288, 13]
Number of token IDs: 11
Input: {'input_ids': tensor([[ 40369,    289,   2267,    811,    304,   2694, 137900,    939,  50568,
            288,     13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 11])


In [11]:
# load the model
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()



e:\Codes\german_tutor\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hridi\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [22]:
with torch.no_grad():
    outputs = model(**inputs)

In [23]:
print(outputs.logits.shape)     # batch size, num of tokens, vocab size

torch.Size([1, 11, 151936])


In [24]:
last_token_logits = outputs.logits[:, -1, :]
print("Last token logits shape:", last_token_logits.shape)
probs = torch.softmax(last_token_logits, dim=-1)
print("Probabilities shape:", probs.shape)

Last token logits shape: torch.Size([1, 151936])
Probabilities shape: torch.Size([1, 151936])


In [25]:
next_token_id = torch.argmax(probs, dim=-1)

print(next_token_id)

next_token = tokenizer.decode(next_token_id)
print("Next token:", next_token)

tensor([25861])
Next token:  Ich
